# Ejercicios 08. La DFT y la FFT

Antes de empezar:

1. Repasa el notebook de teoría `notebooks/08_fft.ipynb`.
2. Ejecuta la celda de preparación, que es la primera celda de código.
3. Busca las marcas `COMPLETA` y cambia los `...` por tu código.
4. Ejecuta las celdas de verificación. Imprimen "Correcto" si tu resultado coincide y "Revisa" si no.
5. Contesta las preguntas de reflexión con tus propias palabras. Es la parte más importante.

Si una celda marca error, lee el último renglón del mensaje; casi siempre ahí está la explicación.
Por ejemplo, `NameError` quiere decir que usaste una variable que no existe o que no ejecutaste
una celda anterior.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## Ejercicio 1. DFT a mano

Calcula a mano la DFT de $x = \{1, 0, -1, 0\}$ ($N = 4$), usando $e^{-j2\pi kn/4} = (-j)^{kn}$.
Escribe el resultado en `X_mano` y compáralo con `pds.dft` y con `np.fft.fft`.

**Tu cálculo a mano:**

$X[0] = $ ...  $X[1] = $ ...  $X[2] = $ ...  $X[3] = $ ...

In [ ]:
x = np.array([1.0, 0.0, -1.0, 0.0])
X_mano = np.array([...])           # COMPLETA
X_dft = ...                        # COMPLETA
X_fft = ...                        # COMPLETA

In [ ]:
pds.comprobar_iguales(X_mano, X_fft, "tu DFT a mano")
pds.comprobar_iguales(X_dft, X_fft, "pds.dft = np.fft.fft")

## Ejercicio 2. Las frecuencias secretas

`../datos/mezcla_tonos.wav` es la suma de **tres tonos** más ruido. Encuentra sus frecuencias y sus amplitudes **relativas**
(divididas entre la del tono más fuerte).

*Algoritmo para encontrar picos:* $k$ es un pico si $|X[k]|$ es mayor que un umbral **y** mayor que sus dos vecinos.
Usa como umbral el 20% del máximo.

In [ ]:
fs, x = pds.cargar_audio("../datos/mezcla_tonos.wav")
f, mag = ...                                        # COMPLETA

umbral = 0.2 * np.max(mag)
frecuencias_pico = []
amplitudes_pico = []
for k in range(1, len(mag) - 1):
    if ...:                                         # COMPLETA
        frecuencias_pico.append(f[k])
        amplitudes_pico.append(mag[k])

amplitudes_relativas = ...                          # COMPLETA
print("Frecuencias:", frecuencias_pico)
print("Amplitudes relativas:", np.round(amplitudes_relativas, 2))

pds.graficar_espectro(f, mag, titulo="Espectro de mezcla_tonos.wav")
plt.axhline(umbral, color="r", linestyle="--")
plt.show()

In [ ]:
pds.comprobar_iguales(frecuencias_pico, [350, 1200, 2750], "las frecuencias son 350, 1200 y 2750 Hz")
pds.comprobar_iguales(amplitudes_relativas, [1.0, 0.6, 0.3], "amplitudes relativas 1, 0.6, 0.3", tolerancia=0.03)

## Ejercicio 3. Resolución en frecuencia

Dos tonos de 440 y 450 Hz ($f_s = 8000$) suenan casi igual. Calcula el espectro con $N = 400, 800, 1600, 3200$
y grafica la zona de 400 a 500 Hz (usa `discreto=True`). Para cada $N$ imprime $\Delta f$.
¿A partir de qué $N$ se ven dos picos separados por un "valle"?

In [ ]:
fs = 8000
lista_N = [400, 800, 1600, 3200]
figura, ejes = plt.subplots(len(lista_N), 1, figsize=(10, 10))
for i in range(len(lista_N)):
    N = lista_N[i]
    delta_f = ...                                            # COMPLETA
    n, t, a = ...                                            # COMPLETA: N muestras de 440 Hz
    n, t, b = ...                                            # COMPLETA: N muestras de 450 Hz
    f, mag = ...                                             # COMPLETA
    pds.graficar_espectro(f, mag, titulo=f"N = {N}, Δf = {delta_f} Hz", ax=ejes[i], discreto=True)
    ejes[i].set_xlim(400, 500)
plt.tight_layout()
plt.show()

**Reflexión:** ¿qué $N$ mínimo separó los tonos? ¿Cuántos **segundos** de audio son? ¿Por qué un afinador de guitarra necesita "escuchar" un rato antes de dar la lectura?

## Ejercicio 4. Nyquist en frecuencia

Muestrea una senoidal de 3500 Hz con $f_s = 5000$ Hz durante 1 s. ¿Dónde **predices** que aparecerá el pico?
Calcula el espectro y comprueba.

In [ ]:
F_predicha = ...                  # COMPLETA (usa pds.frecuencia_alias)
n, t, x = ...                     # COMPLETA
f, mag = ...                      # COMPLETA
F_medida = ...                    # COMPLETA: frecuencia del pico
print("Predicha:", F_predicha, " Medida:", F_medida)

In [ ]:
pds.comprobar(F_predicha == F_medida == 1500, "aparece en 1500 Hz")

## Ejercicio 5. Decodificar un número telefónico (DTMF)

Cuando presionas una tecla de un teléfono, se envía la **suma de dos tonos**: uno de la fila y uno de la columna.

|  | **1209 Hz** | **1336 Hz** | **1477 Hz** |
|---|---|---|---|
| **697 Hz** | 1 | 2 | 3 |
| **770 Hz** | 4 | 5 | 6 |
| **852 Hz** | 7 | 8 | 9 |
| **941 Hz** | * | 0 | # |

`../datos/telefono_dtmf.wav` ($f_s = 8000$) contiene un número de 10 dígitos. Cada dígito es un tono de **0.2 s**
seguido de **0.1 s** de silencio. Tu tarea es descubrir el número.

**Plan:**
1. Escúchalo y grafícalo.
2. Para cada dígito $d = 0, 1, \dots, 9$: recorta el tono (empieza en $d \times 0.3$ s y dura 0.2 s).
3. Calcula su espectro. Busca el pico **por debajo** de 1000 Hz (fila) y el pico **por encima** (columna).
4. Busca en la tabla la fila y columna **más cercanas**.

In [ ]:
fs, x = pds.cargar_audio("../datos/telefono_dtmf.wav")
t = np.arange(len(x)) / fs
pds.graficar_continua(t, x, titulo="Señal DTMF")
plt.show()
pds.reproducir(x, fs)

In [ ]:
filas = np.array([697, 770, 852, 941])
columnas = np.array([1209, 1336, 1477])
teclado = [["1", "2", "3"],
           ["4", "5", "6"],
           ["7", "8", "9"],
           ["*", "0", "#"]]

def decodificar_digito(segmento, fs):
    '''Recibe el pedazo de señal de UN dígito y regresa el carácter.'''
    f, mag = ...                                  # COMPLETA: espectro del segmento

    # Pico de la fila: sólo frecuencias menores a 1000 Hz
    bajas = f < 1000
    F_fila = ...                                  # COMPLETA (pista: f[bajas][np.argmax(mag[bajas])])

    # Pico de la columna: sólo frecuencias mayores a 1000 Hz
    altas = f > 1000
    F_columna = ...                               # COMPLETA

    # Fila y columna más cercanas
    i_fila = np.argmin(np.abs(filas - F_fila))
    i_columna = ...                               # COMPLETA
    return teclado[i_fila][i_columna]

numero = ""
muestras_por_digito = pds.indice_de_muestra(0.3, fs)
muestras_tono = pds.indice_de_muestra(0.2, fs)
for d in range(10):
    inicio = ...                                  # COMPLETA
    segmento = ...                                # COMPLETA
    numero = numero + decodificar_digito(segmento, fs)

print("El número es:", numero)

In [ ]:
pds.comprobar(numero == "5583174260", f"número decodificado: {numero}")

**Reflexión:** con 0.2 s de tono, ¿cuál es la resolución $\Delta f$? ¿Alcanza para distinguir las frecuencias DTMF más cercanas (697 y 770 Hz)? ¿Por qué el sistema DTMF no usa frecuencias que sean múltiplos entre sí?

## Ejercicio 6. Teorema de convolución y el filtro del ECG

**Teorema:** convolución en el tiempo = **multiplicación** en frecuencia: $\;y = x * h \;\Longleftrightarrow\; Y[k] = X[k]\,H[k]$
(si se calculan las DFT con $N \ge L_x + L_h - 1$ puntos).

a) Compruébalo numéricamente con señales aleatorias (`np.fft.fft(x, N)` rellena con ceros hasta $N$).
b) Calcula la **respuesta en frecuencia** $|H(f)|$ del promedio móvil de 6 muestras a $f_s = 360$ Hz: la magnitud de la FFT de $h$ rellenada con ceros a 3600 puntos.
   ¿En qué frecuencias vale **cero**? c) Grafica el espectro del ECG antes y después del filtro.

In [ ]:
# a) Teorema de convolución
x = np.random.randn(50)
h = np.random.randn(10)
N = len(x) + len(h) - 1
Y_tiempo = np.fft.fft(np.convolve(x, h), N)       # FFT de la convolución
Y_frecuencia = ...                                 # COMPLETA: producto de las FFT (con N puntos)

In [ ]:
pds.comprobar_iguales(Y_tiempo, Y_frecuencia, "FFT(x * h) = FFT(x) · FFT(h)")

In [ ]:
# b) Respuesta en frecuencia del promedio móvil
fs_ecg = 360
M = 6
h_pm = np.ones(M) / M                              # h[n] del promedio móvil
N_fft = 3600                                       # muchos puntos -> curva suave (Δf = 0.1 Hz)
H_completa = ...                                   # COMPLETA: np.fft.fft(h_pm, N_fft) rellena h con ceros
mitad = N_fft // 2 + 1                             # sólo de 0 a fs/2
H = ...                                            # COMPLETA: magnitud de las primeras "mitad" muestras
f_h = ...                                          # COMPLETA: pds.eje_frecuencias(...) recortado igual

pds.graficar_espectro(f_h, H, titulo="|H(f)| del promedio móvil de 6 muestras (fs = 360 Hz)")
plt.show()

# c) ECG antes y después
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)
ecg_filtrado = ...                                 # COMPLETA
f1, m1 = pds.espectro(ecg, fs_ecg)
f2, m2 = ...                                       # COMPLETA
plt.figure(figsize=(10, 3.5))
plt.plot(f1, m1, color="0.6", label="original")
plt.plot(f2, m2, "C3", label="filtrado")
plt.legend()
plt.xlabel("Frecuencia (Hz)")
plt.ylim(0, 0.1)
plt.show()

In [ ]:
pds.comprobar(H[f_h == 60][0] < 1e-10, "|H| = 0 en 60 Hz")
pds.comprobar(H[f_h == 120][0] < 1e-10, "|H| = 0 en 120 Hz")
pds.comprobar_iguales(H[0], 1.0, "|H| = 1 en 0 Hz (no altera la componente de CD)")

**Reflexión:** ¿en qué frecuencias se anula el promedio móvil de $M$ muestras? Escribe la fórmula general. ¿Es un pasa-bajas "bueno"?

## Ejercicio 7. El espectro de una nota de piano

Genera la nota de piano del ejercicio 6 del notebook 01 (tecla 40, Do central) y grafica su espectro hasta 2000 Hz.
¿Dónde están los picos? ¿Por qué? Compara con el espectro de un tono puro de la misma frecuencia.

In [ ]:
fs = 16000
F0 = 440 * 2 ** ((40 - 49) / 12)
amplitudes = [1.0, 0.6, 0.35, 0.2, 0.1, 0.05]
n, t, _ = pds.senoidal_digital(F0, fs, 1.5)
nota = np.zeros(len(t))
for i in range(len(amplitudes)):
    n, t, componente = pds.senoidal_digital((i + 1) * F0, fs, 1.5, A=amplitudes[i])
    nota = nota + componente
nota = nota * np.exp(-t / 0.3)

f, mag = ...                                     # COMPLETA
pds.graficar_espectro(f, mag, titulo="Espectro de la nota de piano")
for k in range(1, 8):
    plt.axvline(k * F0, color="r", alpha=0.3)
plt.xlim(0, 2000)
plt.show()

**Reflexión:** ¿por qué los picos se ven "anchos" en lugar de líneas delgadas? (Pista: ¿qué le hicimos a la amplitud con el tiempo?)